In [0]:
from pyspark import pipelines as dp
from pyspark.sql import functions as F

TZ = "America/Sao_Paulo"

# Regra do Data Mart: tempo de resolução usa a primeira ocorrência
# de um status histórico de conclusão.
RESOLUTION_EVENT_STATUSES = [
    "CONCLUIDA",
    "FINALIZADA",
    "COLETADA",
    "COMPLETED",
    "DONE",
    "COLLECTED",
]

# Regra da query 1908: taxa de conclusão usa o status atual da coleta.
CURRENT_COMPLETED_STATUSES = [
    "COMPLETED",
    "DONE",
    "COLLECTED",
    "CONCLUIDA",
    "FINALIZADA",
    "COLETADA",
]


def clean_text(column_name):
    value = F.trim(F.col(column_name).cast("string"))
    return F.when(F.length(value) > 0, value)


@dp.materialized_view(
    name="workspace.silver.occurrence_clean",
    comment="Ocorrências limpas, tipadas e com data local de registro."
)
def occurrence_clean():
    source = spark.read.table("workspace.bronze.incident_raw")

    registered_at_utc = F.col("registered_at").cast("timestamp")
    registered_at_local = F.from_utc_timestamp(registered_at_utc, TZ)

    return source.select(
        F.col("id").cast("string").alias("incident_id"),
        F.col("company_id").cast("string").alias("company_id"),
        F.col("area_id").cast("string").alias("area_id"),
        F.col("waste_type_id").cast("string").alias("waste_type_id"),
        registered_at_local.alias("registered_at_local"),
        F.to_date(registered_at_local).alias("date_key"),
        F.hour(registered_at_local).alias("hour_of_day"),
        F.col("estimated_quantity")
            .cast("decimal(12,2)")
            .alias("estimated_quantity_kg"),
        clean_text("priority").alias("priority"),
        clean_text("status").alias("status"),
        clean_text("contamination_level").alias("contamination_level"),
    )


@dp.materialized_view(
    name="workspace.silver.collection_clean",
    comment=(
        "Coletas limpas; taxa usa status atual e resolução usa "
        "a primeira conclusão histórica."
    )
)
def collection_clean():
    source = spark.read.table("workspace.bronze.collection_raw")

    status_events = (
        spark.read.table("workspace.bronze.collection_status_raw")
        .select(
            F.col("collection_id").cast("string").alias("collection_id"),
            F.upper(F.trim(F.col("status"))).alias("status_norm"),
            F.col("changed_at").cast("timestamp").alias("changed_at_utc"),
        )
    )

    requested_times = source.select(
        F.col("id").cast("string").alias("collection_id"),
        F.col("requested_at").cast("timestamp").alias("requested_at_utc"),
    )

    status_rollup = (
        status_events
        .join(requested_times, "collection_id", "inner")
        .groupBy("collection_id")
        .agg(
            F.max(
                F.when(
                    F.col("status_norm").isin(*RESOLUTION_EVENT_STATUSES),
                    F.lit(1),
                ).otherwise(F.lit(0))
            ).alias("has_completion_event_flag"),
            F.min(
                F.when(
                    F.col("status_norm").isin(*RESOLUTION_EVENT_STATUSES)
                    & (F.col("changed_at_utc") >= F.col("requested_at_utc")),
                    F.col("changed_at_utc"),
                )
            ).alias("completed_event_at_utc")
        )
    )

    incident_keys = (
        spark.read.table("workspace.silver.occurrence_clean")
        .select(
            "incident_id",
            "company_id",
            "area_id",
            "waste_type_id",
        )
    )

    requested_at_utc = F.col("requested_at").cast("timestamp")
    requested_at_local = F.from_utc_timestamp(requested_at_utc, TZ)

    collections = source.select(
        F.col("id").cast("string").alias("collection_id"),
        F.col("incident_id").cast("string").alias("incident_id"),
        F.col("cooperative_id").cast("string").alias("cooperative_id"),
        requested_at_utc.alias("requested_at_utc"),
        requested_at_local.alias("requested_at_local"),
        F.to_date(requested_at_local).alias("requested_date_key"),
        F.from_utc_timestamp(
            F.col("scheduled_at").cast("timestamp"),
            TZ,
        ).alias("scheduled_at_local"),
        F.upper(clean_text("current_status")).alias("current_status"),
        clean_text("collection_type").alias("collection_type"),
        F.col("urgent").cast("boolean").alias("urgent"),
    )

    enriched = (
        collections
        .join(status_rollup, "collection_id", "left")
        .join(incident_keys, "incident_id", "left")
        .withColumn(
            "completed_at_local",
            F.from_utc_timestamp(
                F.col("completed_event_at_utc"),
                TZ,
            ),
        )
        .withColumn(
            "is_currently_completed",
            F.col("current_status").isin(*CURRENT_COMPLETED_STATUSES),
        )
        .withColumn(
            "has_completion_event",
            F.coalesce(F.col("has_completion_event_flag") == 1, F.lit(False)),
        )
        .withColumn(
            "resolution_time_hours",
            F.when(
                F.col("completed_event_at_utc").isNotNull()
                & (F.col("completed_event_at_utc") >= F.col("requested_at_utc")),
                (
                    F.unix_timestamp("completed_event_at_utc")
                    - F.unix_timestamp("requested_at_utc")
                ) / F.lit(3600.0),
            ),
        )
    )

    return enriched.select(
        "collection_id",
        "incident_id",
        "company_id",
        "area_id",
        "waste_type_id",
        "cooperative_id",
        "requested_date_key",
        "requested_at_local",
        "scheduled_at_local",
        "completed_at_local",
        "current_status",
        "collection_type",
        "urgent",
        "is_currently_completed",
        "has_completion_event",
        "resolution_time_hours",
    )